In [433]:
import pandas as pd
import re
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [434]:
from  google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [435]:
df_legis_2002_t1 = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/legis_2002_t1.xls')
df_legis_2007_t1 = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/legis_2007_t1.xls')
df_legis_2012_t1 = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/legis_2012.xls')
df_legis_2017_t1 = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/legis_2017_t1.xlsx', skiprows=3)
#df_legis_2022_t1 = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/legis_2022_t1.txt', sep=';',encoding='latin-1')

df_pres_2002_t1 = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/pres_2002_t1.xls')
df_pres_2007_t1 = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/pres_2007.xls')
df_pres_2012_t1 = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/pres_2012.xls')
df_pres_2017_t1 = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/pres_2017_t1.xls', skiprows=3)
#df_pres_2022_t1 = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/pres_2022_t1.txt', sep=';',encoding='latin-1')

In [436]:
"""df_legis_2022_t1 = pd.read_csv(
    '/content/drive/MyDrive/Colab Notebooks/legis_2022_t1.txt',
    sep=';',
    encoding='latin-1',
    on_bad_lines='warn'
)"""

"df_legis_2022_t1 = pd.read_csv(\n    '/content/drive/MyDrive/Colab Notebooks/legis_2022_t1.txt', \n    sep=';', \n    encoding='latin-1', \n    on_bad_lines='warn'\n)"

In [437]:
mapping_base = {
        'Code du département': 'id_dept',
        'Libellé du département': 'nom_dept',
        'Code de la circonscription': 'id_circo',
        'Libellé de la circonscription': 'nom_circo',
        'Code de la commune': 'id_commune',
        'Libellé de la commune': 'nom_commune',
        'Inscrits': 'nb_inscrits',
        'Abstentions': 'nb_abs',
        '% Abs/Ins': 'tx_abs',
        'Votants': 'nb_votants',
        '% Vot/Ins': 'tx_part',
        'Blancs et nuls': 'nb_blancs_nuls',
        '% BlNuls/Ins': 'tx_blancs_nuls_ins',
        '% BlNuls/Vot': 'tx_blancs_nuls_vot',
        'Blancs':'nb_blancs',
        '% Blancs/Ins':'tx_blancs_ins',
        '% Blancs/Vot':'tx_blancs_vot',
        'Nuls': 'nb_nuls',
        '% Nuls/Ins': 'tx_nuls_ins',
        '% Nuls/Vot':'tx_nuls_vot',
        'Exprimés': 'nb_exprimes',
        '% Exp/Ins': 'tx_exp_ins',
        '% Exp/Vot': 'tx_exp_vot',
        'Sexe': 'cand_sexe',
        'Nom': 'cand_nom',
        'Prénom': 'cand_prenom',
        'Nuance': 'cand_nuance',
        'Voix': 'cand_voix',
        '% Voix/Ins': 'cand_tx_ins',
        '% Voix/Exp': 'cand_tx_exp'
    }

In [438]:

def rename_election_columns(df):

    new_columns = []

    for col in df.columns:
        match = re.match(r"^(.*?)(?:\.(\d+))?$", col)
        base_name = match.group(1)
        suffix = match.group(2)

        if base_name in mapping_base:
            new_name = mapping_base[base_name]
            if suffix:
                new_columns.append(f"{new_name}_{suffix}")
            elif base_name in ['Sexe', 'Nom', 'Prénom', 'Nuance', 'Voix', '% Voix/Ins', '% Voix/Exp']:
                new_columns.append(f"{new_name}_0")
            else:
                new_columns.append(new_name)
        else:
            new_columns.append(col.lower().replace(' ', '_'))

    df.columns = new_columns
    return df

In [538]:
mapping_politique = {
    # --- PARTIS & SIGLES ---
    'LO': 'Extrême Gauche', 'LCR': 'Extrême Gauche', 'EXG': 'Extrême Gauche',
    'COM': 'Gauche', 'SOC': 'Gauche', 'FG': 'Gauche', 'FI': 'Gauche',
    'RDG': 'Gauche', 'PRG': 'Gauche', 'DVG': 'Gauche', 'PREP': 'Gauche',
    'VEC': 'Écologie', 'ECO': 'Écologie',
    'UDF': 'Centre', 'UDFD': 'Centre', 'CEN': 'Centre', 'MDM': 'Centre', 'REM': 'Centre',
    'UMP': 'Droite', 'LR': 'Droite', 'DVD': 'Droite', 'DL': 'Droite', 'UDI': 'Droite',
    'PRV': 'Droite', 'NCE': 'Droite', 'ALLI': 'Droite', 'MAJ': 'Droite',
    'DLF': 'Droite', 'MPF': 'Droite', 'RPF': 'Droite',
    'CPNT': 'Droite',
    'FN': 'Extrême Droite', 'MNR': 'Extrême Droite', 'EXD': 'Extrême Droite',
    'REG': 'Régionalisme', 'DIV': 'Divers', 'AUT': 'Divers',

    # --- CANDIDATS (Noms complets tels que dans tes arrays) ---
    # Extrême Gauche
    'LAGUILLER ARLETTE': 'Extrême Gauche', 'LAGUILLER Arlette': 'Extrême Gauche',
    'BESANCENOT OLIVIER': 'Extrême Gauche', 'BESANCENOT Olivier': 'Extrême Gauche',
    'POUTOU Philippe': 'Extrême Gauche', 'ARTHAUD Nathalie': 'Extrême Gauche',
    'GLUCKSTEIN DANIEL': 'Extrême Gauche', 'SCHIVARDI Gérard': 'Extrême Gauche',

    # Gauche
    'JOSPIN LIONEL': 'Gauche', 'HUE ROBERT': 'Gauche', 'BUFFET Marie-George': 'Gauche',
    'MÉLENCHON Jean-Luc': 'Gauche', 'HOLLANDE François': 'Gauche',
    'ROYAL Ségolène': 'Gauche', 'HAMON Benoît': 'Gauche', 'TAUBIRA CHRISTIANE': 'Gauche',

    # Écologie
    'MAMERE NOEL': 'Écologie', 'VOYNET Dominique': 'Écologie', 'JOLY Eva': 'Écologie',

    # Centre
    'BAYROU FRANCOIS': 'Centre', 'BAYROU François': 'Centre',
    'MACRON Emmanuel': 'Centre',

    # Droite
    'CHIRAC JACQUES': 'Droite', 'SARKOZY Nicolas': 'Droite', 'FILLON François': 'Droite',
    'MADELIN ALAIN': 'Droite', 'BOUTIN CHRISTINE': 'Droite',
    'CHEVENEMENT JEAN-PIERRE': 'Gauche',
    'DE VILLIERS Philippe': 'Droite', 'de VILLIERS Philippe': 'Droite',
    'DUPONT-AIGNAN Nicolas': 'Droite', 'SAINT-JOSSE JEAN': 'Droite',
    'NIHOUS Frédéric': 'Droite', 'BOVÉ José': 'Gauche',
    'LASSALLE Jean': 'Divers',

    # Extrême Droite
    'LE PEN JEAN-MARIE': 'Extrême Droite', 'LE PEN Jean-Marie': 'Extrême Droite',
    'LE PEN Marine': 'Extrême Droite', 'MEGRET BRUNO': 'Extrême Droite',

    # Autres
    'CHEMINADE Jacques': 'Divers', 'ASSELINEAU François': 'Divers',
    'LEPAGE CORINNE': 'Centre'
}



In [539]:
def clean_election_df(df):
  res = df[df['id_dept'] == 35].copy()
  #res=df.copy()
  if 'nb_blancs_nuls' not in res.columns:
    if 'nb_blancs' in res.columns and 'nb_nuls' in res.columns:
      res['nb_blancs_nuls'] = res['nb_blancs'] + res['nb_nuls']

  res = res.drop(columns=['nb_blancs', 'nb_nuls'], errors='ignore')
  res = res.drop(columns=['nom_dept', 'nom_commune', 'nb_expri', 'nom_circo', 'id_circo'], errors='ignore')

  forbidden_keywords = ['tx', 'panneau', 'sexe','unnamed']
  res = res[[col for col in res.columns if not any(key in col for key in forbidden_keywords)]]

  return res

In [540]:
def create_nuance_df(df):
  res=df.copy()

  if res['cand_nuance'].isnull().all():
      res['cand_nuance'] = res['cand_nom']+' '+res['cand_prenom']

  res = res.drop(columns=['cand_nom', 'cand_prenom'], errors='ignore')

  res = res.dropna(subset=['cand_nuance'])

  return res

In [557]:
def mapping_orientation(df):
  df['cand_orientation'] = df['cand_nuance'].map(mapping_politique)
  df = df.drop(columns=['cand_nuance'], errors='ignore')
  return df

In [558]:
def pivot_results(df):
    id_vars = ['id_dept', 'id_commune', 'nb_inscrits',
               'nb_abs', 'nb_votants', 'nb_blancs_nuls']

    df_long = pd.wide_to_long(
        df,
        stubnames=['cand_nuance', 'cand_nom', 'cand_prenom', 'cand_voix'],
        i=id_vars,
        j='candidat_id',
        sep='_'
    ).reset_index()

    return df_long

In [559]:
def traitement_election_df(df):
  res = (
      df
      .pipe(rename_election_columns)
      .pipe(clean_election_df)
      .pipe(pivot_results)
      .pipe(create_nuance_df)
      .pipe(mapping_orientation)
  )
  return res

In [563]:
df_legis_2002_t1_propre = traitement_election_df(df_legis_2002_t1)

In [564]:
df_legis_2007_t1_propre = traitement_election_df(df_legis_2007_t1)

In [565]:
df_legis_2012_t1_propre = traitement_election_df(df_legis_2012_t1)

In [566]:
df_legis_2017_t1_propre = traitement_election_df(df_legis_2017_t1)

In [567]:
df_pres_2002_t1_propre = traitement_election_df(df_pres_2002_t1)

In [568]:
df_pres_2007_t1_propre = traitement_election_df(df_legis_2007_t1)

In [569]:
df_pres_2012_t1_propre = traitement_election_df(df_legis_2012_t1)

In [570]:
df_pres_2017_t1_propre = traitement_election_df(df_legis_2017_t1)

In [572]:
df_legis_2002_t1_propre.to_csv('df_legis_2002_t1_propre.csv', index=False)

In [573]:
df_legis_2007_t1_propre.to_csv('df_legis_2007_t1_propre.csv', index=False)

In [574]:
df_legis_2012_t1_propre.to_csv('df_legis_2012_t1_propre.csv', index=False)

In [575]:
df_legis_2017_t1_propre.to_csv('df_legis_2017_t1_propre.csv', index=False)

In [576]:
df_pres_2002_t1_propre.to_csv('pres_2002_t1_propre.csv', index=False)

In [577]:
df_pres_2007_t1_propre.to_csv('pres_2007_t1_propre.csv', index=False)

In [578]:
df_pres_2012_t1_propre.to_csv('pres_2012_t1_propre.csv', index=False)

In [579]:
df_pres_2017_t1_propre.to_csv('pres_2017_t1_propre.csv', index=False)